# Repeat a procedural Frozen Lake task

[repeat-task-gym](https://github.com/micahr234/repeat-task-gym) groups episodes into a task. Mouse-gym provides a separate `RepeatTaskTracker` to interpret the resulting stream. This notebook builds a `RepeatTaskEnv` around [Procedural Frozen Lake](https://github.com/micahr234/procedural-frozenlake) and hands that Gymnasium env to `make_env` through `env_fn`.

`RepeatTaskEnv` puts episode end inside the observation (`episode_terminated`, `episode_truncated`) and uses Gymnasium `terminated` / `truncated` for the task. Mouse-gym forwards the observation, those flags, and `info`. `tracker.episodes` collects inner episodes and `tracker.tasks` collects completed tasks. The tracker is separate from the env and is explicitly updated after every output. The next `step()` is a reset frame that starts another task (`tracker.step_index == 0`). There is no public `reset()`.

Episode resets pass `regenerate_map: False`, so both episodes of a task share one lake. `EnvConfig.reset_options` is forwarded on every reset mouse-gym issues. Those task resets pass `regenerate_map: True`, which overrides the episode option for the first episode only, so each new task gets a new map.


In [ ]:
import gymnasium as gym
from procedural_frozenlake import ensure_registered

from mouse_gym import EnvConfig, RepeatTaskTracker, make_env
from repeat_task_gym import RepeatTaskEnv

ensure_registered()


def make_task():
    lake = gym.make(
        "Procedural-FrozenLake-v1",
        map_seed=0,
        width=5,
        height=5,
        min_border=1,
        max_border=1,
        shoreline_jaggedness=0,
        max_episode_steps=8,
        emit_map=True,
    )
    return RepeatTaskEnv(
        lake,
        max_task_episodes=2,
        episode_seed_mode="per_task",
        episode_reset_options={"regenerate_map": False},
    )


env = make_env(
    EnvConfig(
        seed=0,
        seed_mode="constant",
        name="repeat-frozen-lake",
        env_fn=make_task,
        reset_options={"regenerate_map": True},
    )
)


## Roll through two tasks

Each task is two episodes of one generated lake (`max_task_episodes=2`). `episode_seed_mode="per_task"` reuses one reset seed for both, so the start tile matches. `max_episode_steps=8` ends a wandering episode at the step limit.

Between episodes, `RepeatTaskEnv.step(None)` returns its own boundary step: reward `0`, episode flags false, and Gymnasium flags false. That step passes `regenerate_map: False`, so the board in `info["map"]` stays the same. Mouse-gym does not treat it as a reset. `tracker.step_index` keeps increasing, and the step is included in the task length recorded later.

The task ends when Gymnasium `terminated` or `truncated` is true. In the first task that is the second episode (`truncated=True`). The recorded length counts every step since mouse-gym's reset frame, including `RepeatTaskEnv`'s between-episode step: `17` is eight steps, that boundary step, and eight more steps. The following `step()` is mouse-gym's reset frame (`step_index == 0`). It passes `regenerate_map: True` and draws a different board.

In this seeded run the first task times out both episodes, so the return is `0.0` and the length is `17` (eight steps, the reset frame, eight more steps). The second task times out, then reaches the goal (`reward=1.0`, `episode_terminated=True`); its length is `13`. The loop prints task resets, episode-end steps, and the reset frame between episodes.


In [ ]:
tracker = RepeatTaskTracker()
env.action_space.seed(0)
seen_tasks = 0
show_reset_frame = False
try:
    for _ in range(200):
        action = env.action_space.sample() if tracker.action_requested else None
        output = env.step(action)
        tracker.update(output=output)
        observation, reward, terminated, truncated, info = output
        episode_terminated = bool(observation["episode_terminated"])
        episode_truncated = bool(observation["episode_truncated"])
        episode_end = episode_terminated or episode_truncated
        if tracker.step_index == 0 or episode_end or show_reset_frame:
            print(
                f"step={tracker.step_index} "
                f"reward={reward:.1f} "
                f"terminated={terminated} "
                f"truncated={truncated} "
                f"episode_terminated={episode_terminated} "
                f"episode_truncated={episode_truncated} "
                f"board={info['map']['board']} "
                f"returns={[task.reward for task in tracker.tasks]} "
                f"lengths={[task.length for task in tracker.tasks]}"
            )
        show_reset_frame = episode_end and not (terminated or truncated)
        if terminated or truncated:
            seen_tasks += 1
        if seen_tasks >= 2 and tracker.step_index == 0:
            break
finally:
    env.close()
